# EMIT / Sentinel-2 — CNMF Fusion (GT Generation)

Runs the Coupled Non-negative Matrix Factorization (CNMF) hyperspectral–multispectral
fusion to produce 32-band ground-truth imagery at 10 m for SR training.

**Inputs**: tile pairs on Drive (produced by `Pairs_Extract.ipynb`), R² CSV from `Color_Matching.ipynb`  
**Outputs**: CNMF-fused GeoTIFFs, per-tile R² CSV, diagnostic plots, per-band analysis

**Note**: CNMF is significantly slower than SFIM (~5–30 s per tile vs ~1 s). Plan for
longer runtimes with large tile sets.

## 1. Setup

In [ ]:
import os, textwrap
from google.colab import userdata

os.environ["GH_USER"] = userdata.get("GH_USER")
os.environ["GH_TOKEN"] = userdata.get("GH_TOKEN")

askpass_path = "/tmp/gh_askpass.sh"
with open(askpass_path, "w") as f:
    f.write(textwrap.dedent("""\
        #!/bin/sh
        case \"$1\" in
          *Username*) echo \"$GH_USER\" ;;
          *Password*) echo \"$GH_TOKEN\" ;;
          *) echo \"\" ;;
        esac
    """))
os.chmod(askpass_path, 0o700)
os.environ["GIT_ASKPASS"] = askpass_path
os.environ["GIT_TERMINAL_PROMPT"] = "0"

!git clone https://github.com/incredet/HyperSpectralSuperResolution.git
%cd /content/HyperSpectralSuperResolution/

In [ ]:
!pip install -q numpy scipy scikit-learn rasterio matplotlib tqdm joblib opencv-python-headless

In [ ]:
import json, re, time
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

import rasterio

from spectral.cnmf import cnmf_fuse_tile
from spectral.s2_to_emit import plot_spectral_match, plot_r2_spectrum
from documentation.report_builder import R2Aggregator, plot_r2_histogram, plot_r2_per_band

print("All imports OK.")

## 2. Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 3. Parameters

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  USER PARAMETERS — edit these before running
# ═══════════════════════════════════════════════════════════════════

DRIVE_ROOT = Path("/content/drive/Shareddrives/HyperResData/EMIT_S-2_Matches")
RUN_TAG    = "2026-03-24"

# General tile parameters
TILE_SCALE     = 6
SCALE_FACTOR   = 10_000.0   # DN → reflectance
NODATA_VAL     = 65535      # EMIT nodata sentinel

# CNMF parameters
MAX_ENDMEMBERS = 20         # Maximum number of endmembers
INNER_ITERS    = 200        # Max inner NMF iterations
OUTER_ITERS    = 1          # Max outer CNMF iterations (1 is typical for real data)

# R² filtering — use regression R² from Color_Matching.ipynb
R2_THRESHOLD   = 0.75

# Plotting
SHOW_PLOTS       = False    # set True for interactive exploration
PLOT_EVERY_N     = 50       # save a diagnostic plot every N tiles
PLOT_FIRST_N     = 3        # always plot the first N tiles per pair

# ═══════════════════════════════════════════════════════════════════

## 4. Load R² CSV and build tile list

Uses `r2_all_tiles.csv` from `Color_Matching.ipynb` to select tiles with
good spectral agreement (R² ≥ threshold), and constructs file paths
directly from the CSV — no globbing on Drive.

In [ ]:
drive_base = DRIVE_ROOT / RUN_TAG
r2_csv_path = drive_base / "r2_all_tiles.csv"

r2_df = pd.read_csv(r2_csv_path)
print(f"R\u00b2 CSV: {len(r2_df)} tiles total")
print(f"R\u00b2 >= {R2_THRESHOLD}: {(r2_df['r2_mean'] >= R2_THRESHOLD).sum()} tiles pass")

# Filter
r2_pass = r2_df[r2_df["r2_mean"] >= R2_THRESHOLD].copy()
r2_pass = r2_pass.sort_values("r2_mean", ascending=False).reset_index(drop=True)

# Build file paths from CSV columns
def build_tile_paths(row):
    aoi_slug = row["aoi_slug"]
    pair_id  = row["pair_id"]
    tile_idx = int(row["tile_idx"])
    tile_name = f"{pair_id}_tile{tile_idx:03d}"
    tiles_dir = drive_base / aoi_slug / pair_id / "tiles"
    return {
        "hs_path":  tiles_dir / f"{tile_name}_emit_b32.tif",
        "ms_path":  tiles_dir / f"{tile_name}_s2.tif",
        "tile_name": tile_name,
        "pair_dir": drive_base / aoi_slug / pair_id,
    }

tile_list = []
for _, row in r2_pass.iterrows():
    paths = build_tile_paths(row)
    paths["aoi_slug"] = row["aoi_slug"]
    paths["pair_id"]  = row["pair_id"]
    paths["tile_idx"] = int(row["tile_idx"])
    paths["r2_regression"] = row["r2_mean"]
    tile_list.append(paths)

print(f"Tile list built: {len(tile_list)} tiles")
print(f"AOIs: {len(set(t['aoi_slug'] for t in tile_list))}")
print(f"Pairs: {len(set(t['pair_id'] for t in tile_list))}")

## 5. Run CNMF fusion

Processes each tile: reads EMIT-b32 + S2, runs pure-Python CNMF,
saves the fused GeoTIFF to `{pair_dir}/CNMF/`, and collects per-tile
R² and timing metrics. Skips tiles that already have output.

**Expected time**: ~5–30 s per tile depending on convergence.
First few tiles run with `verbose=True` so you can monitor progress.

In [ ]:
all_cnmf_rows = []   # per-tile results
n_processed = 0
n_skipped = 0
n_existed = 0
n_missing = 0
tic_total = time.time()

for i, tile in enumerate(tqdm(tile_list, desc="CNMF fusion")):
    hs_path = tile["hs_path"]
    ms_path = tile["ms_path"]
    pair_dir = tile["pair_dir"]
    tile_name = tile["tile_name"]

    cnmf_dir = pair_dir / "CNMF"
    out_path = cnmf_dir / f"{tile_name}_cnmf.tif"

    # Skip if already exists
    if out_path.exists():
        n_existed += 1
        continue

    # Check files exist
    if not hs_path.exists() or not ms_path.exists():
        n_missing += 1
        continue

    # Run CNMF (verbose for first few tiles)
    use_verbose = (n_processed < 3)
    tic = time.time()
    try:
        result = cnmf_fuse_tile(
            hs_path, ms_path, out_path,
            scale_factor=SCALE_FACTOR,
            nodata_val=NODATA_VAL,
            max_endmembers=MAX_ENDMEMBERS,
            inner_iters=INNER_ITERS,
            outer_iters=OUTER_ITERS,
            verbose=use_verbose,
        )
    except Exception as e:
        tqdm.write(f"  ERROR {tile_name}: {e}")
        all_cnmf_rows.append({
            "aoi_slug": tile["aoi_slug"],
            "pair_id": tile["pair_id"],
            "tile_idx": tile["tile_idx"],
            "status": "ERROR",
            "r2_cnmf_mean": None,
            "r2_regression": tile["r2_regression"],
            "time_s": time.time() - tic,
        })
        n_skipped += 1
        continue

    elapsed = time.time() - tic

    row = {
        "aoi_slug": tile["aoi_slug"],
        "pair_id": tile["pair_id"],
        "tile_idx": tile["tile_idx"],
        "status": result["status"],
        "r2_cnmf_mean": result["r2_mean"],
        "r2_regression": tile["r2_regression"],
        "time_s": elapsed,
    }
    # Store CNMF info
    if result.get("info"):
        row["cnmf_M"] = result["info"].get("M")
        row["cnmf_rmse_h"] = result["info"].get("rmse_h")
        row["cnmf_rmse_m"] = result["info"].get("rmse_m")
    if result.get("r2_per_band"):
        for bi, v in enumerate(result["r2_per_band"]):
            row[f"r2_cnmf_band_{bi:02d}"] = v
    all_cnmf_rows.append(row)

    # Diagnostic plots
    should_plot = (n_processed < PLOT_FIRST_N) or (n_processed % PLOT_EVERY_N == 0)
    if should_plot and result["status"] == "OK":
        plots_dir = pair_dir / "plots"
        plots_dir.mkdir(parents=True, exist_ok=True)

        try:
            plot_spectral_match(
                str(ms_path), str(out_path), str(hs_path),
                title_suffix=f"{tile_name} (CNMF)",
                r2_mean=result["r2_mean"],
                save_path=str(plots_dir / f"{tile_name}_cnmf_match.png"),
                show=SHOW_PLOTS,
            )
        except Exception as e:
            tqdm.write(f"  Plot error {tile_name}: {e}")

    n_processed += 1

elapsed_total = time.time() - tic_total
print(f"\nDone in {elapsed_total/60:.1f} min")
print(f"  Processed: {n_processed}")
print(f"  Already existed: {n_existed}")
print(f"  Missing files: {n_missing}")
print(f"  Errors/skipped: {n_skipped}")
if n_processed > 0:
    print(f"  Avg time: {elapsed_total / n_processed:.2f}s/tile")

## 6. Save CNMF R² results

In [ ]:
cnmf_df = pd.DataFrame(all_cnmf_rows)

cnmf_csv_path = drive_base / "r2_cnmf_tiles.csv"
cnmf_df.to_csv(cnmf_csv_path, index=False)

ok_df = cnmf_df[cnmf_df["status"] == "OK"]
print(f"CNMF R\u00b2 CSV saved: {cnmf_csv_path}  ({len(cnmf_df)} tiles)")
print(f"  OK: {len(ok_df)}, Errors: {len(cnmf_df) - len(ok_df)}")
print(f"\nCNMF R\u00b2 summary (OK tiles):")
print(f"  Mean:   {ok_df['r2_cnmf_mean'].mean():.4f}")
print(f"  Median: {ok_df['r2_cnmf_mean'].median():.4f}")
print(f"  Min:    {ok_df['r2_cnmf_mean'].min():.4f}")
print(f"  Max:    {ok_df['r2_cnmf_mean'].max():.4f}")

print(f"\nRegression R\u00b2 vs CNMF R\u00b2 comparison:")
print(f"  Regression mean: {ok_df['r2_regression'].mean():.4f}")
print(f"  CNMF mean:       {ok_df['r2_cnmf_mean'].mean():.4f}")

# CNMF-specific diagnostics
if "cnmf_M" in ok_df.columns:
    print(f"\nCNMF endmembers (M):")
    print(f"  Mean: {ok_df['cnmf_M'].mean():.1f}  Min: {ok_df['cnmf_M'].min()}  Max: {ok_df['cnmf_M'].max()}")
if "cnmf_rmse_h" in ok_df.columns:
    print(f"  RMSE_h mean: {ok_df['cnmf_rmse_h'].mean():.6f}")
    print(f"  RMSE_m mean: {ok_df['cnmf_rmse_m'].mean():.6f}")

## 7. Global diagnostics

### 7a. R² distribution and comparison with regression

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (1) CNMF R² histogram
axes[0].hist(ok_df["r2_cnmf_mean"], bins=30, edgecolor="black", alpha=0.7, color="steelblue")
axes[0].axvline(ok_df["r2_cnmf_mean"].median(), color="red", ls="--",
                label=f"median={ok_df['r2_cnmf_mean'].median():.3f}")
axes[0].set_xlabel("CNMF R\u00b2 (mean over bands)")
axes[0].set_ylabel("Count")
axes[0].set_title("CNMF R\u00b2 distribution")
axes[0].legend()

# (2) Regression R² vs CNMF R² scatter
axes[1].scatter(ok_df["r2_regression"], ok_df["r2_cnmf_mean"],
                s=5, alpha=0.3, color="steelblue")
lims = [min(ok_df["r2_regression"].min(), ok_df["r2_cnmf_mean"].min()),
        max(ok_df["r2_regression"].max(), ok_df["r2_cnmf_mean"].max())]
axes[1].plot(lims, lims, "--", color="orange", linewidth=1, label="y = x")
axes[1].set_xlabel("Regression R\u00b2")
axes[1].set_ylabel("CNMF R\u00b2")
axes[1].set_title("Regression vs CNMF R\u00b2")
axes[1].legend()

# (3) Per-AOI box plot of CNMF R²
aoi_order = ok_df.groupby("aoi_slug")["r2_cnmf_mean"].median().sort_values(ascending=False).index
ok_sorted = ok_df.set_index("aoi_slug").loc[aoi_order].reset_index()
ok_sorted.boxplot(column="r2_cnmf_mean", by="aoi_slug", ax=axes[2], rot=90)
axes[2].set_title("CNMF R\u00b2 by AOI")
axes[2].set_xlabel("")
axes[2].set_ylabel("CNMF R\u00b2")
plt.suptitle("")

plt.tight_layout()
fig.savefig(str(drive_base / "cnmf_r2_global_summary.png"), dpi=150, bbox_inches="tight")
plt.show()

### 7b. Per-band R² analysis

In [ ]:
# Collect per-band R² columns
r2_band_cols = [c for c in ok_df.columns if c.startswith("r2_cnmf_band_")]

if r2_band_cols:
    r2_per_band = ok_df[r2_band_cols].values  # (N_tiles, B_hs)
    r2_band_mean = np.nanmean(r2_per_band, axis=0)
    r2_band_std  = np.nanstd(r2_per_band, axis=0)
    r2_band_min  = np.nanmin(r2_per_band, axis=0)
    r2_band_p25  = np.nanpercentile(r2_per_band, 25, axis=0)
    r2_band_p75  = np.nanpercentile(r2_per_band, 75, axis=0)
    bands = np.arange(len(r2_band_cols))

    # Try to get wavelengths from a sample tile
    sample_hs = tile_list[0]["hs_path"]
    try:
        with rasterio.open(sample_hs) as ds:
            wl_list = []
            for b in range(1, ds.count + 1):
                bt = ds.tags(b)
                w = bt.get("wavelength") or bt.get("WAVELENGTH")
                if w: wl_list.append(float(w))
            x_axis = np.array(wl_list) if len(wl_list) == len(bands) else bands
            x_label = "Wavelength (nm)" if len(wl_list) == len(bands) else "Band index"
    except:
        x_axis = bands
        x_label = "Band index"

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # (1) Mean ± std per band
    axes[0].plot(x_axis, r2_band_mean, color="steelblue", linewidth=1.5, label="Mean")
    axes[0].fill_between(x_axis, r2_band_mean - r2_band_std, r2_band_mean + r2_band_std,
                         alpha=0.2, color="steelblue", label="\u00b11\u03c3")
    axes[0].plot(x_axis, r2_band_min, "--", color="tomato", linewidth=0.8, label="Min")
    axes[0].set_xlabel(x_label)
    axes[0].set_ylabel("CNMF R\u00b2")
    axes[0].set_title("Per-band CNMF R\u00b2 (all tiles)")
    axes[0].legend()
    axes[0].set_ylim(0, 1.05)
    axes[0].grid(True, alpha=0.3)

    # (2) Box plot per band
    axes[1].boxplot(r2_per_band, positions=range(len(bands)),
                    widths=0.6, showfliers=False,
                    medianprops={"color": "steelblue", "linewidth": 1.5})
    axes[1].set_xlabel("Band index")
    axes[1].set_ylabel("CNMF R\u00b2")
    axes[1].set_title("Per-band R\u00b2 box plot")
    axes[1].set_ylim(0, 1.05)
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    fig.savefig(str(drive_base / "cnmf_r2_per_band.png"), dpi=150, bbox_inches="tight")
    plt.show()

    # Print worst/best bands
    worst_band = np.argmin(r2_band_mean)
    best_band = np.argmax(r2_band_mean)
    print(f"Best band:  {best_band} (R\u00b2={r2_band_mean[best_band]:.4f})")
    print(f"Worst band: {worst_band} (R\u00b2={r2_band_mean[worst_band]:.4f})")
else:
    print("No per-band R\u00b2 columns found (all tiles may have been skipped).")

### 7c. Timing analysis

In [ ]:
ok_times = ok_df["time_s"].dropna()

if len(ok_times) > 0:
    print(f"Timing per tile (OK tiles):")
    print(f"  Mean:   {ok_times.mean():.3f}s")
    print(f"  Median: {ok_times.median():.3f}s")
    print(f"  Min:    {ok_times.min():.3f}s")
    print(f"  Max:    {ok_times.max():.3f}s")
    print(f"  Total:  {ok_times.sum()/60:.1f} min")

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(ok_times, bins=30, edgecolor="black", alpha=0.7, color="steelblue")
    ax.axvline(ok_times.median(), color="red", ls="--",
               label=f"median={ok_times.median():.2f}s")
    ax.set_xlabel("Time per tile (s)")
    ax.set_ylabel("Count")
    ax.set_title("CNMF processing time distribution")
    ax.legend()
    plt.tight_layout()
    fig.savefig(str(drive_base / "cnmf_timing.png"), dpi=150, bbox_inches="tight")
    plt.show()

## 8. Summary statistics

In [ ]:
# Final counts
total_cnmf = len(list(drive_base.glob("*/*/CNMF/*_cnmf.tif")))
print(f"Total CNMF GeoTIFFs on Drive: {total_cnmf}")
print(f"R\u00b2 CSV: {cnmf_csv_path}")

# Per-AOI summary
if len(ok_df) > 0:
    summary = ok_df.groupby("aoi_slug").agg(
        n_tiles=("r2_cnmf_mean", "count"),
        r2_mean=("r2_cnmf_mean", "mean"),
        r2_median=("r2_cnmf_mean", "median"),
        r2_min=("r2_cnmf_mean", "min"),
        time_mean=("time_s", "mean"),
    ).sort_values("r2_mean", ascending=False)

    print(f"\nPer-AOI summary:")
    print(summary.to_string())